# 02 — Feature Engineering

Build **one row per customer** with time-windowed behavioral features per merchant category.

For each of the 14 merchant categories × each time window (6m, 12m), compute:
- `n_txn_{cat}_{w}m`        — number of transactions
- `amt_{cat}_{w}m`          — total spend ($)
- `avg_spend_{cat}_{w}m`    — average spend per transaction ($)

Plus for each category (all-time):
- `consec_months_{cat}`     — max consecutive months with ≥1 transaction (loyalty signal)

Plus overall customer metrics:
- avg transaction size, spend volatility, active months, unique categories visited

Reference date for windows = the last transaction date in the dataset (Dec 2020).
Each window looks back from that reference date.

In [ ]:
import pandas as pd
import numpy as np
import json
import pathlib
from dateutil.relativedelta import relativedelta

pathlib.Path('data/processed').mkdir(parents=True, exist_ok=True)

df = pd.read_csv('data/raw/fraudTrain.csv', index_col=0,
                 parse_dates=['trans_date_trans_time'])
df['merchant'] = df['merchant'].str.replace('^fraud_', '', regex=True)
df['year_month'] = df['trans_date_trans_time'].dt.to_period('M')

CATEGORIES = sorted(df['category'].unique())
REFERENCE_DATE = df['trans_date_trans_time'].max()
WINDOWS = [6, 12]   # months to look back

print(f'Loaded {len(df):,} rows, {df["cc_num"].nunique()} customers')
print(f'Reference date (end of data): {REFERENCE_DATE.date()}')
print(f'Time windows: {WINDOWS} months')
print(f'Categories  : {CATEGORIES}')

## Feature Group 1 — Time-Windowed Per-Category Features

For each customer × each category × each window:
- `n_txn_{cat}_{w}m`
- `amt_{cat}_{w}m`
- `avg_spend_{cat}_{w}m`

In [ ]:
windowed_frames = []

for w in WINDOWS:
    cutoff = REFERENCE_DATE - relativedelta(months=w)
    df_w = df[df['trans_date_trans_time'] > cutoff]

    for cat in CATEGORIES:
        df_cat = df_w[df_w['category'] == cat]

        n_txn = df_cat.groupby('cc_num')['amt'].count().rename(f'n_txn_{cat}_{w}m')
        amt   = df_cat.groupby('cc_num')['amt'].sum().rename(f'amt_{cat}_{w}m')
        # avg_spend: avoid div-by-zero by computing from the grouped values
        avg   = df_cat.groupby('cc_num')['amt'].mean().rename(f'avg_spend_{cat}_{w}m')

        windowed_frames.extend([n_txn, amt, avg])

# Combine all windowed features, indexed by cc_num
windowed = pd.concat(windowed_frames, axis=1).fillna(0)
print(f'Windowed feature matrix: {windowed.shape}  '
      f'({len(CATEGORIES)} cats × {len(WINDOWS)} windows × 3 metrics = '
      f'{len(CATEGORIES)*len(WINDOWS)*3} features)')
windowed.head(3)

## Feature Group 2 — Consecutive Months per Category (Loyalty Signal)

For each customer × category: **max number of consecutive calendar months** with at least 1 transaction.

Example: shopped at `grocery_pos` in Jan, Feb, Mar, May → max streak = 3

In [ ]:
def max_consecutive_months(month_periods):
    """Given a list of Period('M') values, return the max consecutive month streak."""
    if len(month_periods) == 0:
        return 0
    sorted_months = sorted(set(month_periods))
    max_streak, streak = 1, 1
    for i in range(1, len(sorted_months)):
        # Check if consecutive: difference of 1 month
        diff = (sorted_months[i] - sorted_months[i-1]).n
        if diff == 1:
            streak += 1
            max_streak = max(max_streak, streak)
        else:
            streak = 1
    return max_streak

consec_frames = []
for cat in CATEGORIES:
    df_cat = df[df['category'] == cat]
    consec = df_cat.groupby('cc_num')['year_month'].apply(
        max_consecutive_months
    ).rename(f'consec_months_{cat}')
    consec_frames.append(consec)

consec_features = pd.concat(consec_frames, axis=1).fillna(0)
print(f'Consecutive-month features: {consec_features.shape}')
consec_features.head(3)

## Feature Group 3 — Overall Customer Metrics

In [ ]:
overall = df.groupby('cc_num').apply(lambda g: pd.Series({
    'total_txn_count'     : len(g),
    'total_spend'         : g['amt'].sum(),
    'avg_txn_amt'         : g['amt'].mean(),
    'std_txn_amt'         : g['amt'].std(),
    'max_txn_amt'         : g['amt'].max(),
    'pct_high_value'      : (g['amt'] > 200).mean(),
    'n_unique_categories' : g['category'].nunique(),
    'n_unique_merchants'  : g['merchant'].nunique(),
    'active_months'       : g['year_month'].nunique(),
    'avg_days_between_txn': (
        g['trans_date_trans_time'].sort_values().diff().dropna().dt.days.mean()
        if len(g) > 1 else np.nan
    ),
}), include_groups=False)

print(f'Overall features: {overall.shape}')
overall.head(3)

## Combine All Features

In [ ]:
features = pd.concat([windowed, consec_features, overall], axis=1)

# Drop customers with fewer than 10 total transactions
before = len(features)
features = features[features['total_txn_count'] >= 10]
print(f'Dropped {before - len(features)} customers with < 10 transactions')
print(f'Final feature table: {features.shape[0]} customers × {features.shape[1]} features')

# Handle missing values (e.g. std on single-txn customers)
missing = features.isnull().sum()
if missing.any():
    print(f'Filling {missing.sum()} missing values with column medians')
    features = features.fillna(features.median(numeric_only=True))
else:
    print('No missing values.')

print('\nSample features (first 3 rows, first 10 cols):')
features.iloc[:3, :10]

In [ ]:
# Save
features.to_parquet('data/processed/customer_features.parquet')

with open('data/processed/feature_names.json', 'w') as f:
    json.dump(list(features.columns), f, indent=2)

print(f'Saved customer_features.parquet  ({features.shape[0]} rows × {features.shape[1]} cols)')
print(f'Saved feature_names.json')

## Quick Sanity Check — Feature Preview

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Show mean 12-month transaction count per category across all customers
cols_12m_n = [f'n_txn_{cat}_12m' for cat in CATEGORIES]
mean_txn = features[cols_12m_n].mean().sort_values(ascending=False)
mean_txn.index = [c.replace('n_txn_','').replace('_12m','') for c in mean_txn.index]

plt.figure(figsize=(12,5))
mean_txn.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Average # Transactions per Category (last 12 months)')
plt.ylabel('Avg transaction count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('outputs/cluster_plots/feat_avg_txn_12m.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Show mean consecutive months per category — identifies which categories have loyal shoppers
cols_consec = [f'consec_months_{cat}' for cat in CATEGORIES]
mean_consec = features[cols_consec].mean().sort_values(ascending=False)
mean_consec.index = [c.replace('consec_months_','') for c in mean_consec.index]

plt.figure(figsize=(12,5))
mean_consec.plot(kind='bar', color='coral', edgecolor='white')
plt.title('Average Max Consecutive Months per Category (loyalty signal)')
plt.ylabel('Avg consecutive months')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('outputs/cluster_plots/feat_consec_months.png', dpi=150, bbox_inches='tight')
plt.show()